## Imports

In [1]:
import os
import pandas as pd
import psycopg2

PG_HOST = os.getenv("PGHOST", "localhost")
PG_PORT = int(os.getenv("PGPORT", "5432"))
PG_USER = os.getenv("PGUSER", "mlflow")
PG_PASSWORD = os.getenv("PGPASSWORD", "Eesee8th")

def connect_db(dbname: str):
    return psycopg2.connect(
        dbname=dbname,
        user=PG_USER,
        password=PG_PASSWORD,
        host=PG_HOST,
        port=PG_PORT,
    )

conn = connect_db("mlflow")

conn.autocommit = True

In [2]:
query = """select now() as today"""

with conn.cursor() as cur:
    cur.execute(query)
    t = cur.fetchall()
    
print(t)

[(datetime.datetime(2026, 3, 15, 22, 8, 33, 50216, tzinfo=datetime.timezone.utc),)]


## Base query

In [3]:
query = """
SELECT pid, usename, state, wait_event_type, wait_event, query
FROM pg_stat_activity
WHERE state <> 'idle'
ORDER BY query_start;
"""

pd.read_sql_query(query, conn)

/tmp/ipykernel_118804/1751318874.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(query, conn)


,pid,usename,state,wait_event_type,wait_event,query
0,1015,mlflow,active,None,None,"\nSELECT pid, usename, state, wait_event_type,..."


In [39]:
query = """
SELECT pg_terminate_backend(2236)
FROM pg_stat_activity
WHERE datname = 'mlflow'
  AND pid != pg_backend_pid();
"""

# with conn.cursor() as cur:
#     cur.execute("SET statement_timeout = '10s'")
#     cur.execute(query)
#     print(cur.statusmessage)

In [4]:
# All databases + sizes
pd.read_sql_query("""
    SELECT datname AS database,
           pg_size_pretty(pg_database_size(datname)) AS size,
           datallowconn AS connectable
    FROM pg_database
    WHERE datname NOT IN ('template0', 'template1', 'postgres')
    ORDER BY datname
""", conn)

/tmp/ipykernel_118804/359811252.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query("""


,database,size,connectable
0,agent042,7807 kB,True
1,airflow,11 MB,True
2,mlflow,9447 kB,True


In [ ]:
# Roles / users
pd.read_sql_query("""
    SELECT rolname AS role,
           rolsuper AS superuser,
           rolcreaterole AS can_create_role,
           rolcreatedb AS can_create_db,
           rolcanlogin AS can_login
    FROM pg_roles
    WHERE rolname NOT LIKE 'pg_%'
    ORDER BY rolname
""", conn)

In [ ]:
# Active connections
pd.read_sql_query("""
    SELECT datname AS database,
           usename AS user,
           application_name,
           state,
           count(*) AS connections
    FROM pg_stat_activity
    WHERE datname IS NOT NULL
    GROUP BY datname, usename, application_name, state
    ORDER BY connections DESC
""", conn)

## 3. MLflow database (`mlflow`)

In [6]:
# Tables
pd.read_sql_query("""
    SELECT tablename AS table,
           pg_size_pretty(pg_total_relation_size(quote_ident(tablename))) AS total_size
    FROM pg_tables
    WHERE schemaname = 'public'
    ORDER BY pg_total_relation_size(quote_ident(tablename)) DESC
""", conn)

/tmp/ipykernel_118804/380977222.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query("""


,table,total_size
0,log,288 kB
1,task_instance,224 kB
2,dag_run,160 kB
3,job,128 kB
4,serialized_dag,120 kB
5,dag,104 kB
6,xcom,96 kB
7,dag_code,88 kB
8,ab_user,80 kB
9,session,80 kB


In [ ]:
# Experiments
pd.read_sql_query("""
    SELECT experiment_id, name, lifecycle_stage,
           to_timestamp(creation_time / 1000) AS created_at
    FROM experiments
    ORDER BY creation_time DESC
    LIMIT 20
""", conn)

In [ ]:
# Recent runs (last 20)
pd.read_sql_query("""
    SELECT r.run_uuid,
           e.name AS experiment,
           r.status,
           to_timestamp(r.start_time / 1000) AS started_at,
           to_timestamp(r.end_time   / 1000) AS ended_at
    FROM runs r
    JOIN experiments e USING (experiment_id)
    ORDER BY r.start_time DESC
    LIMIT 20
""", conn)

In [ ]:
# Registered models
pd.read_sql_query("""
    SELECT name,
           to_timestamp(creation_time / 1000) AS created_at,
           to_timestamp(last_updated_time / 1000) AS last_updated
    FROM registered_models
    ORDER BY last_updated_time DESC
""", conn)

In [ ]:
# Model versions
pd.read_sql_query("""
    SELECT name, version, current_stage, status,
           run_id,
           to_timestamp(creation_time / 1000) AS created_at
    FROM model_versions
    ORDER BY creation_time DESC
""", conn)

## 4. Airflow database (`airflow`)

In [5]:
# Switch connection to Airflow DB
conn.close()
conn = connect_db("airflow")

# Tables
pd.read_sql_query("""
    SELECT tablename AS table,
           pg_size_pretty(pg_total_relation_size(quote_ident(tablename))) AS total_size
    FROM pg_tables
    WHERE schemaname = 'public'
    ORDER BY pg_total_relation_size(quote_ident(tablename)) DESC
""", conn)

/tmp/ipykernel_118804/952971551.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query("""


,table,total_size
0,log,288 kB
1,task_instance,224 kB
2,dag_run,160 kB
3,job,128 kB
4,serialized_dag,120 kB
5,dag,104 kB
6,xcom,96 kB
7,dag_code,88 kB
8,ab_user,80 kB
9,session,80 kB


In [ ]:
# DAGs
pd.read_sql_query("""
    SELECT dag_id, is_active, is_paused, fileloc,
           last_parsed_time, next_dagrun
    FROM dag
    ORDER BY dag_id
""", conn)

In [ ]:
# Recent DAG runs (last 30)
pd.read_sql_query("""
    SELECT dag_id, run_id, state, run_type,
           execution_date, start_date, end_date
    FROM dag_run
    ORDER BY start_date DESC NULLS LAST
    LIMIT 30
""", conn)

In [ ]:
# Failed task instances (last 20)
pd.read_sql_query("""
    SELECT dag_id, task_id, run_id, state,
           start_date, end_date, duration,
           try_number
    FROM task_instance
    WHERE state = 'failed'
    ORDER BY start_date DESC NULLS LAST
    LIMIT 20
""", conn)

## 5. Application database (`agent042`)

In [7]:
# Switch connection to Application DB
conn.close()
conn = connect_db("agent042")

# Tables + sizes
pd.read_sql_query("""
    SELECT tablename AS table,
           pg_size_pretty(pg_total_relation_size(quote_ident(tablename))) AS total_size
    FROM pg_tables
    WHERE schemaname = 'public'
    ORDER BY pg_total_relation_size(quote_ident(tablename)) DESC
""", conn)

/tmp/ipykernel_118804/3976356846.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query("""


,table,total_size
0,chat_messages,72 kB
1,users,48 kB
2,chat_sessions,32 kB
3,eval_runs,32 kB


In [8]:
query = """
select *
from eval_runs
"""

pd.read_sql_query(query, conn)

/tmp/ipykernel_118804/339666776.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(query, conn)


,id,created_at,finished_at,status,task,dataset_name,metric_name,metric_value,base_model,adapter_name,...,score_threshold,qdrant_snapshot_id,dataset_dvc_hash,reranking_strategy,judge_model,bert_score_model,temperature,max_tokens,extra,error_message
0,f4dd2fe2-a8f7-4ed2-8bce-d670ec620438,2026-03-15 22:06:58.451184+00:00,2026-03-15 22:06:58.520190+00:00,completed,chat,hotpotqa,rouge_l,0.006768,/models/Qwen/Qwen3-0.6B,None,...,None,None,None,None,gemini-2.0-flash,microsoft/deberta-xlarge-mnli,0.0,512,{},None


In [ ]:
# Column listing for all tables
pd.read_sql_query("""
    SELECT table_name, column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_schema = 'public'
    ORDER BY table_name, ordinal_position
""", conn)